# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [22]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
    ("deepseek", "deepseek-chat", "DeepSeek Chat")
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free', 'DeepSeek Chat']


In [16]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1",
    "deepseek": "https://api.deepseek.com/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY"),
    "deepseek": os.getenv("DEEPSEEK_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [18]:
# varianta minimala

# fara functie
client = make_client("deepseek") #make_client=functie creata mai jos
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="deepseek",
    model="deepseek-chat",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un tip de inteligență artificială antrenată pe cantități masive de text pentru a înțelege și genera limbaj uman. Practic, funcționează prin prezicerea cuvântului următor dintr-o secvență, bazându-se pe tiparele și informațiile învățate din datele de antrenament.
Un LLM (Large Language Model) este un model avansat de inteligență artificială specializat în înțelegerea și generarea textului într-un mod similar cu cel uman, fiind antrenat pe cantități masive de date. Acesta funcționează prin prezicerea cuvintelor următoare dintr-o secvență, bazându-se pe tiparele statistice și relațiile dintre cuvinte învățate în timpul antrenamentului.


In [27]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    # Pentru DeepSeek care nu intelege response_format
    if provider == "deepseek":
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature
        )
        return response.choices[0].message.content
    
    # 
    else:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            response_format={"type": "json_object"}  # doar dacă ai nevoie
        )
        return response.choices[0].message.content

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

In [30]:
from openai import RateLimitError, APIError, AuthenticationError
import json
import time

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None, max_retries=3):
    """
    Trimite un prompt la model. Poate returna text simplu sau JSON structurat.
    
    Args:
        provider: "gemini", "deepseek", sau "openrouter"
        model: ID-ul modelului
        prompt: mesajul utilizatorului
        system: mesajul de sistem (opțional)
        temperature: 0-1 pentru creativitate
        json_schema: schema JSON pentru răspuns structurat (opțional)
        max_retries: număr de reîncercări la rate limit
    
    Returns:
        text sau dict (dacă json_schema e specificat)
    """
    
    client = make_client(provider)
    
    # Construiește mesajele
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    extra_args = {}
    
    # Configurare response_format în funcție de provider și json_schema
    if json_schema:
        # Pentru providerii care suportă JSON Schema
        if provider in ["gemini", "openrouter"]:
            extra_args["response_format"] = {
                "type": "json_schema",
                "json_schema": json_schema
            }
        elif provider == "deepseek":
            # DeepSeek suportă doar json_object, nu json_schema
            extra_args["response_format"] = {"type": "json_object"}
            # Adaugă instrucțiune explicită în prompt
            messages[-1]["content"] += "\n\nRespond ONLY with valid JSON. Do not include any explanatory text."
    elif provider in ["gemini", "openrouter"]:
        # Dacă nu avem json_schema dar e un provider care suportă JSON
        pass  # nu adăugăm response_format automat
    
    # Reîncercare automată la rate limit
    for attempt in range(max_retries):
        try:
            # Pentru DeepSeek fără json_schema, nu folosim extra_args
            if provider == "deepseek" and not json_schema:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    temperature=temperature
                )
            else:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    temperature=temperature,
                    **extra_args
                )
            
            text = response.choices[0].message.content.strip()
            
            # Parsează JSON dacă e cerut
            if json_schema:
                # Încearcă să parseze JSON-ul
                try:
                    return json.loads(text)
                except json.JSONDecodeError:
                    # Dacă DeepSeek nu returnează JSON valid, încearcă să extragă
                    if provider == "deepseek":
                        import re
                        json_match = re.search(r'\{.*\}', text, re.DOTALL)
                        if json_match:
                            return json.loads(json_match.group())
                    return {"error": "Invalid JSON response", "raw_response": text}
            
            return text
        
        except RateLimitError as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # exponential backoff: 1, 2, 4 secunde
                print(f"[Rate limit la {provider}, reîncerc în {wait_time}s...]")
                time.sleep(wait_time)
            else:
                return f"[Eroare: quota/rate limit pentru modelul {model}. {e}]"
        
        except AuthenticationError:
            return f"[Eroare: API key invalidă sau lipsă pentru {provider}. Verifică .env.]"
        
        except APIError as e:
            # Pentru DeepSeek, unele erori pot fi rezolvate fără json_schema
            if provider == "deepseek" and "response_format" in str(e) and json_schema:
                print(f"[DeepSeek nu suportă json_schema, reîncerc fără...]")
                # Reîncercăm fără json_schema
                json_schema = None  # dezactivează json_schema
                extra_args = {}
                continue
            return f"[Eroare API: {e}]"
        
        except Exception as e:
            return f"[Eroare: {type(e).__name__} — {e}]"
    
    return "[Eroare: Număr maxim de reîncercări depășit]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [23]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

--- Gemini 2.5 Flash ---
Politica românească a fost marcată de o frecventă schimbare a guvernelor și de o tendință către formarea unor coaliții largi, atipice. Aceasta a culminat cu actuala coaliție PSD-PNL, pe fondul unei fragmentări crescute a peisajului politic și apariției de noi actori.

--- OpenRouter Free ---
În ultimii cinci ani,România a consolidat alianțele cu NATO și UE, participând activ la misiuni în Black Sea și susținând sancțiunile împotriva Rupei. Simultan, a intensificat relațiile bilaterale cu Moldova, Ucraina și Republica Moldova, menținând o politică de detergență regională și asigurând securitatea frontală.

--- DeepSeek Chat ---
În ultimii cinci ani, scena politică românească a fost marcată de alternanța la guvernare între PNL și PSD, inclusiv formarea unor coaliții succesive. De asemenea, au avut loc alegeri prezidențiale și parlamentare, iar procesul de aderare la S

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [34]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Critic, cinic
Emoție dominantă: Frustrare, neîncredere
Țintă principală: Clasa politică
Populism: da

--- Gemini 2.5 Flash ---
Ton: Acuzator, cinic.
Emoție dominantă: Frustrare.
Țintă principală: Clasa politică.
Populism: da

--- OpenRouter Free ---
Ton: cenzurat, acusator  
Emoție dominantă: frustrație/îngrijorare  
Țintă principală: politici  
Populism: da

--- DeepSeek Chat ---
Ton: Cinic și acuzator.
Emoție dominantă: Frustrare și neîncredere.
Țintă principală: Clasa politică în ansamblu.
Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [35]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [ ]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
[Rate limit la gemini, reîncerc în 1s...]
[Rate limit la gemini, reîncerc în 2s...]
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite. Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 6.449250668s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate

In [39]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

# Definim maparea pentru DeepSeek (transformă ce returnează DeepSeek în schema corectă)
DEEPSEEK_FIELD_MAPPING = {
    "sentiment": "ton",           # DeepSeek "sentiment" → schema "ton"
    "target": "tinta_principala", # DeepSeek "target" → schema "tinta_principala"
    "topic": "explicatie_scurta", # DeepSeek "topic" → schema "explicatie_scurta" (opțional)
    "emotion": "emotie_dominanta" # DeepSeek "emotion" → schema "emotie_dominanta"
}

# Valori implicite pentru câmpurile lipsă
DEFAULT_VALUES = {
    "ton": "neutru",
    "emotie_dominanta": "neutru",
    "tinta_principala": "nespecificat",
    "populism": False,
    "explicatie_scurta": "Nicio explicație disponibilă"
}

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE  # folosește schema ta completă
    )

    # Pentru DeepSeek, mapează și completează câmpurile lipsă
    if provider == "deepseek" and isinstance(rezultat, dict):
        
        # Aplică maparea
        mapped_result = {}
        for old_key, value in rezultat.items():
            if old_key in DEEPSEEK_FIELD_MAPPING:
                new_key = DEEPSEEK_FIELD_MAPPING[old_key]
                mapped_result[new_key] = value
            else:
                # Păstrează câmpurile care deja sunt corecte
                mapped_result[old_key] = value
        
        # Completează câmpurile lipsă cu valori implicite
        for required_field in ["ton", "emotie_dominanta", "tinta_principala", "populism", "explicatie_scurta"]:
            if required_field not in mapped_result:
                mapped_result[required_field] = DEFAULT_VALUES[required_field]
        
        # Asigură-te că valorile respectă enum-urile
        if mapped_result["ton"] not in ["pozitiv", "negativ", "neutru"]:
            mapped_result["ton"] = "neutru"
        
        if mapped_result["emotie_dominanta"] not in ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]:
            mapped_result["emotie_dominanta"] = "neutru"
        
        rezultat = mapped_result
        print("DeepSeek:", rezultat)
    
    print("Output:", rezultat)


--- Gemini 2.5 Flash Lite ---
[Rate limit la gemini, reîncerc în 1s...]
[Rate limit la gemini, reîncerc în 2s...]
Output: [Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite. Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 1.514633605s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/

KeyboardInterrupt: 

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [10]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea de noi alegeri, ceea ce implică o perioadă de incertitudine politică și o posibilă reconfigurare a forțelor politice. Această decizie poate influența legitimitatea procesului electoral și poate genera dezbateri privind respectarea normelor constituționale.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea de noi alegeri, ceea ce implică o perioadă de incertitudine politică și o posibilă reorganizare a forțelor politice. Acest proces poate influența legitimitatea instituțiilor și poate necesita ajustări ale proceselor electorale pentru viitor.

temperature=1.2:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

[ Gemini 2.5 Flash ]

temperature=0.1:
[Eroare API: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently expe

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da  | da  | da  | da  | Am atins destul de repede quota/rate limit, dar Gemini 2.5 Flash a continuat sa functioneze |
| OpenRouter Free |  parțial | da  | da  |  nu | Este cel mai slow si uneori scrie in engleza |
| Deepseek | da  | da  | da  | da  | DeepSeek are un comportament diferit față de Gemini în ceea ce privește formatarea outputului structurat |
### Decizie
**Model principal ales:** Gemini 2.5 Flash 
**Model de rezervă:**  Deepseek
**Temperature recomandată:**  0.3
**De ce am ales acest model?**  
Gemini 2.5 Flash este cel mai potrivit pentru taskurile din cadrul proiectului, intrucât funcționează bine pentru adnotarea comentariilor politice în limba română. Răspunde corect în română, respectă instrucțiunile și schema JSON. Și a continuat să funcționeze chiar și după ce Flash Lite a atins limita, limita care se poate gestiona prin prin folosirea modelului de rezervă (DeepSeek).

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [ ]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.3

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales